In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # nada sale a internet salvo Databricks si se configura
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))   # los notebooks importan de src/, nunca al revés

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

# 06 · Optimización con seguimiento (MLflow)

- **Finalistas:** LightGBM y XGBoost.
- **Entidades reservadas:** 20 % de los **clientes** de entrenamiento (semilla 42) que la búsqueda no ve.
- **Búsqueda:** Optuna (TPE), 30 ensayos LightGBM y 20 XGBoost, fijados antes de empezar (≈ 1 min por cada 6 ensayos);
  objetivo **PR-AUC** en 5 pliegues por cliente dentro del 80 % restante. Cada ensayo es una ejecución anidada en MLflow
  con todas las métricas.
- **Regla escrita antes de mirar:** se adoptan los parámetros optimizados solo si el intervalo del 95 % de la diferencia
  de PR-AUC (optimizado − partida) en los clientes reservados queda **entero por encima de 0**. Si no, se usan los de partida.

In [2]:
import json
import joblib
from proyecto.config import GROUP_COL, TARGET, MIN_PRECISION, REPORTS_DIR
from proyecto.data.split import group_cv
from proyecto.models.train import build_pipeline, candidates, evaluate_folds, split_xy, summarize
from proyecto.tracking.mlflow_utils import configure_mlflow, log_run, set_experiment

train = joblib.load(ROOT / "data" / "processed" / "train.joblib")
X, y = split_xy(train)
cv = group_cv(5)   # los MISMOS 5 pliegues por cliente para todo (semilla 42)
donde = configure_mlflow()
print(f"entrenamiento: {len(train):,} créditos · mora {y.mean():.1%} · MLflow: {donde}")

entrenamiento: 36,872 créditos · mora 15.8% · MLflow: local


In [3]:
from sklearn.model_selection import GroupShuffleSplit
from proyecto.models.tuning import START_PARAMS, MODELS, tune, bootstrap_metric_difference, bootstrap_metric_ci
from sklearn.metrics import average_precision_score
IDEAS = tuple(json.loads((REPORTS_DIR / "04_features_finales.json").read_text())["ideas"])
fin = json.loads((REPORTS_DIR / "05_finalistas.json").read_text())["finalistas"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
busq_idx, res_idx = next(gss.split(X, y, groups=X[GROUP_COL]))
Xb, yb, Xr, yr = X.iloc[busq_idx], y[busq_idx], X.iloc[res_idx], y[res_idx]
assert not set(Xb[GROUP_COL]) & set(Xr[GROUP_COL])
print(f"búsqueda: {len(Xb):,} créditos · reservados: {len(Xr):,} créditos de {Xr[GROUP_COL].nunique():,} clientes")

búsqueda: 29,468 créditos · reservados: 7,404 créditos de 5,545 clientes


In [4]:
set_experiment("libranzaradar-06-optimizacion")
ENSAYOS = {"lightgbm": 30, "xgboost": 20}
estudios = {}
for nombre in fin:
    estudios[nombre] = tune(nombre, Xb, yb, group_cv(5), n_trials=ENSAYOS[nombre], ideas=IDEAS, metric="pr_auc",
                            storage=ROOT / "data" / "processed" / f"optuna_{nombre}.db")
    print(f"{nombre}: mejor PR-AUC en la búsqueda {estudios[nombre].best_value:.4f} (optimista)")

2026/10/31 11:32:51 INFO mlflow.tracking.fluent: Experiment with name 'libranzaradar-06-optimizacion' does not exist. Creating a new experiment.


[I 2026-10-31 11:32:51,848] A new study created in RDB with name: libranzaradar-xgboost-pr_auc


xgboost: mejor PR-AUC en la búsqueda 0.1787 (optimista)


lightgbm: mejor PR-AUC en la búsqueda 0.1787 (optimista)


In [5]:
filas = []
for nombre, est in estudios.items():
    df_t = est.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
    df_t["modelo"] = nombre
    filas.append(df_t)
ensayos = pd.concat(filas)
cols = ["modelo", "number", "value", "user_attrs_pr_auc_std", "user_attrs_recall_at_p", "user_attrs_roc_auc", "user_attrs_log_loss"]
ensayos[cols].sort_values("value", ascending=False).round(4).to_csv(REPORTS_DIR / "06_optimizacion_tabla.csv", index=False)
ensayos[cols].sort_values("value", ascending=False).head(8).round(4)

,modelo,number,value,user_attrs_pr_auc_std,user_attrs_recall_at_p,user_attrs_roc_auc,user_attrs_log_loss
16,xgboost,16,0.1787,0.0038,0.1690,0.5364,0.4360
22,lightgbm,22,0.1787,0.0050,0.1966,0.5360,0.4360
29,lightgbm,29,0.1786,0.0044,0.1714,0.5360,0.4360
23,lightgbm,23,0.1785,0.0046,0.2009,0.5353,0.4360
18,xgboost,18,0.1785,0.0039,0.1891,0.5357,0.4360
5,xgboost,5,0.1785,0.0039,0.1602,0.5354,0.4360
18,lightgbm,18,0.1784,0.0043,0.1677,0.5361,0.4360
14,xgboost,14,0.1784,0.0055,0.1529,0.5352,0.4362


## Comprobación honesta en los clientes reservados

In [6]:
comp, mejores = [], {}
for nombre, est in estudios.items():
    make = MODELS[nombre][1]
    p_ini = build_pipeline(make(START_PARAMS[nombre]), ideas=IDEAS).fit(Xb, yb).predict_proba(Xr)[:, 1]
    p_opt = build_pipeline(make(est.best_params), ideas=IDEAS).fit(Xb, yb).predict_proba(Xr)[:, 1]
    dif, lo, hi = bootstrap_metric_difference(yr, p_opt, p_ini, Xr[GROUP_COL], average_precision_score, n_boot=500)
    mejores[nombre] = {"partida": START_PARAMS[nombre], "optimizados": est.best_params,
                       "pr_auc_reservado_partida": average_precision_score(yr, p_ini),
                       "pr_auc_reservado_optimizado": average_precision_score(yr, p_opt),
                       "diferencia": dif, "ic95": [lo, hi], "adoptar_optimizados": bool(lo > 0)}
    comp.append({"modelo": nombre, "PR-AUC partida": round(average_precision_score(yr, p_ini), 4),
                 "PR-AUC optimizado": round(average_precision_score(yr, p_opt), 4), "diferencia": round(dif, 4),
                 "IC 95 %": f"[{lo:.4f}; {hi:.4f}]", "¿adoptar optimizados?": "sí" if lo > 0 else "no (dentro del ruido)"})
comp = pd.DataFrame(comp)
comp.to_csv(REPORTS_DIR / "06_comprobacion_reservados.csv", index=False)
(REPORTS_DIR / "06_mejores_parametros.json").write_text(json.dumps(mejores, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
comp

,modelo,PR-AUC partida,PR-AUC optimizado,diferencia,IC 95 %,¿adoptar optimizados?
0,xgboost,0.1705,0.1736,0.0030,[-0.0028; 0.0089],no (dentro del ruido)
1,lightgbm,0.1710,0.1738,0.0027,[-0.0035; 0.0098],no (dentro del ruido)


In [7]:
# Modelo que pasa a validación: el finalista con mejor PR-AUC en reservados, con los parámetros que dicta la regla
ganador = max(mejores, key=lambda n: max(mejores[n]["pr_auc_reservado_partida"], mejores[n]["pr_auc_reservado_optimizado"]))
info = mejores[ganador]
parametros = info["optimizados"] if info["adoptar_optimizados"] else info["partida"]
eleccion = {"modelo": ganador, "parametros": parametros, "origen": "optimizados" if info["adoptar_optimizados"] else "partida"}
(REPORTS_DIR / "06_eleccion.json").write_text(json.dumps(eleccion, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
eleccion

{'modelo': 'lightgbm',
 'parametros': {'n_estimators': 300,
  'learning_rate': 0.03,
  'num_leaves': 15,
  'min_child_samples': 100,
  'subsample': 0.8,
  'colsample_bytree': 0.8,
  'reg_lambda': 1.0},
 'origen': 'partida'}